In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np

# ============================================================
# Experiment configuration
# ============================================================

SEED = 42
INJECTION_RATIO = 0.0005   # 0.05%
SCENARIO = "inconsistent"

VALID_SCENARIOS = {
    "extreme",
    "noise_strong",
    "noise_moderate",
    "noise_mild",
    "inconsistent"
}

if SCENARIO not in VALID_SCENARIOS:
    raise ValueError(
        f"Unknown scenario: {SCENARIO}. "
        f"Expected one of {sorted(VALID_SCENARIOS)}."
    )

# ============================================================
# Paths
# ============================================================

INJECTED_DATA_DIR = Path(
    "03_Injected_Proposed_Results_Seed_42\Injected_Datasets_with_0.05%_Injection_Rate"
)

BASELINE_ARTIFACTS_DIR = Path(
    "02_Baseline_Results_Seed_42"
)

ratio_tag = "ratio_0p00050"

INJECTED_BASELINE_RESULTS_DIR = Path(
    "02_Injected_Baseline_Results_Seed_42"
)

SCENARIO_RESULTS_DIR = (
    INJECTED_BASELINE_RESULTS_DIR
    / ratio_tag
    / SCENARIO
)

SCENARIO_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Scenario-file mapping
# ============================================================

scenario_files = {
    "extreme": "X_injected_extreme.npy",
    "noise_strong": "X_injected_noise_strong.npy",
    "noise_moderate": "X_injected_noise_moderate.npy",
    "noise_mild": "X_injected_noise_mild.npy",
    "inconsistent": "X_injected_inconsistent.npy"
}

scenario_path = (
    INJECTED_DATA_DIR
    / scenario_files[SCENARIO]
)

inject_idx_path = (
    INJECTED_DATA_DIR
    / "inject_indices.npy"
)

injected_labels_path = (
    INJECTED_DATA_DIR
    / "injection_ground_truth_labels.npy"
)

scaler_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_scaler.pkl"
)

# ============================================================
# Verify required files
# ============================================================

required_paths = [
    scenario_path,
    inject_idx_path,
    injected_labels_path,
    scaler_path
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

# ============================================================
# Load data
# ============================================================

X_clean = np.load("X.npy")
y = np.load("y.npy")

X_injected_raw = np.load(
    scenario_path
)

inject_idx = np.load(
    inject_idx_path
).astype(
    np.int64,
    copy=False
)

injected_ground_truth = np.load(
    injected_labels_path
).astype(
    np.uint8,
    copy=False
)

n_samples, n_features = X_clean.shape

# ============================================================
# Validation
# ============================================================

if X_injected_raw.shape != X_clean.shape:
    raise ValueError(
        "Injected and clean matrices have different shapes: "
        f"{X_injected_raw.shape} and {X_clean.shape}"
    )

if inject_idx.ndim != 1:
    raise ValueError(
        "inject_indices.npy must contain a one-dimensional array."
    )

if injected_ground_truth.shape != (n_samples,):
    raise ValueError(
        "Unexpected injection-ground-truth shape: "
        f"{injected_ground_truth.shape}"
    )

if np.any(inject_idx < 0) or np.any(inject_idx >= n_samples):
    raise IndexError(
        "inject_indices.npy contains out-of-range indices."
    )

if np.unique(inject_idx).size != len(inject_idx):
    raise ValueError(
        "inject_indices.npy contains duplicate indices."
    )

ground_truth_indices = np.flatnonzero(
    injected_ground_truth > 0
)

if not np.array_equal(
    np.sort(inject_idx),
    ground_truth_indices
):
    raise ValueError(
        "inject_indices.npy does not match "
        "injection_ground_truth_labels.npy."
    )

actual_injection_ratio = (
    len(inject_idx) / n_samples
)

print("Scenario:", SCENARIO)
print("Clean shape:", X_clean.shape)
print("Injected shape:", X_injected_raw.shape)
print("Injected samples:", f"{len(inject_idx):,}")
print(
    "Actual injection rate:",
    f"{100 * actual_injection_ratio:.6f}%"
)

# ============================================================
# Apply the frozen clean baseline scaler
# ============================================================

baseline_scaler = joblib.load(
    scaler_path
)

X_injected_scaled = baseline_scaler.transform(
    X_injected_raw
).astype(
    np.float32,
    copy=False
)

if not np.isfinite(X_injected_scaled).all():
    raise ValueError(
        "Scaled injected data contains NaN or infinite values."
    )

print("Scaled shape:", X_injected_scaled.shape)
print("Scaled dtype:", X_injected_scaled.dtype)
print("Scaled min:", float(X_injected_scaled.min()))
print("Scaled max:", float(X_injected_scaled.max()))
print("Scaled mean:", float(X_injected_scaled.mean()))
print("Scaled std:", float(X_injected_scaled.std()))

# ============================================================
# Save experiment metadata
# ============================================================

np.save(
    SCENARIO_RESULTS_DIR
    / "inject_indices.npy",
    inject_idx
)

np.save(
    SCENARIO_RESULTS_DIR
    / "injection_ground_truth_labels.npy",
    injected_ground_truth
)

np.savez(
    SCENARIO_RESULTS_DIR
    / "experiment_metadata.npz",
    seed=np.int64(SEED),
    scenario=np.asarray(SCENARIO),
    injection_ratio_configured=np.float64(INJECTION_RATIO),
    injection_ratio_actual=np.float64(actual_injection_ratio),
    total_samples=np.int64(n_samples),
    total_features=np.int64(n_features),
    total_injected=np.int64(len(inject_idx)),
    injected_file=np.asarray(
        scenario_files[SCENARIO]
    ),
    injected_data_folder=np.asarray(
        str(INJECTED_DATA_DIR)
    )
)

print(
    "Results folder:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Validate and preprocess selected injected scenario
# Frozen clean StandardScaler only
# ============================================================

# ------------------------------------------------------------
# Validate raw clean and injected matrices
# ------------------------------------------------------------

def validate_finite_matrix(
    name,
    matrix
):
    nan_count = int(
        np.isnan(matrix).sum()
    )

    inf_count = int(
        np.isinf(matrix).sum()
    )

    print(
        f"{name} NaN count:",
        f"{nan_count:,}"
    )

    print(
        f"{name} Inf count:",
        f"{inf_count:,}"
    )

    if nan_count > 0 or inf_count > 0:
        raise ValueError(
            f"{name} contains NaN or infinite values."
        )


validate_finite_matrix(
    "Clean raw matrix",
    X_clean
)

validate_finite_matrix(
    f"{SCENARIO} raw matrix",
    X_injected_raw
)

# ------------------------------------------------------------
# Verify that only selected rows differ
# ------------------------------------------------------------

injected_mask = np.zeros(
    n_samples,
    dtype=bool
)

injected_mask[
    inject_idx
] = True

non_injected_idx = np.flatnonzero(
    ~injected_mask
)

# A complete comparison of all non-injected rows is large but
# still methodologically useful. Compare in batches to avoid
# creating another full difference matrix.
def verify_unchanged_non_injected_rows(
    clean_matrix,
    injected_matrix,
    non_injected_indices,
    batch_size=100_000
):
    for start_idx in range(
        0,
        len(non_injected_indices),
        batch_size
    ):
        current_indices = non_injected_indices[
            start_idx:
            start_idx + batch_size
        ]

        if not np.array_equal(
            clean_matrix[current_indices],
            injected_matrix[current_indices]
        ):
            raise ValueError(
                "At least one non-injected row differs from "
                "the clean dataset."
            )


verify_unchanged_non_injected_rows(
    clean_matrix=X_clean,
    injected_matrix=X_injected_raw,
    non_injected_indices=non_injected_idx
)

print(
    "Non-injected rows match the clean dataset."
)

# ------------------------------------------------------------
# Load scaler fitted in clean baseline notebook
# ------------------------------------------------------------

baseline_scaler = joblib.load(
    BASELINE_ARTIFACTS_DIR
    / "Baseline_scaler.pkl"
)

if getattr(
    baseline_scaler,
    "n_features_in_",
    None
) != n_features:
    raise ValueError(
        "Saved scaler expects "
        f"{baseline_scaler.n_features_in_} features, "
        f"but injected data contains {n_features}."
    )

# ------------------------------------------------------------
# Transform only — never fit on injected data
# ------------------------------------------------------------

X_injected_scaled = baseline_scaler.transform(
    X_injected_raw
).astype(
    np.float32,
    copy=False
)

validate_finite_matrix(
    f"{SCENARIO} scaled matrix",
    X_injected_scaled
)

print(
    "\nScaled injected shape:",
    X_injected_scaled.shape
)

print(
    "Scaled injected dtype:",
    X_injected_scaled.dtype
)

print(
    "Global minimum:",
    float(X_injected_scaled.min())
)

print(
    "Global maximum:",
    float(X_injected_scaled.max())
)

print(
    "Global mean:",
    float(X_injected_scaled.mean())
)

print(
    "Global standard deviation:",
    float(X_injected_scaled.std())
)

# ------------------------------------------------------------
# Per-feature diagnostics
# ------------------------------------------------------------

print("\nPer-feature scaled statistics:")

for feature_idx in range(n_features):

    feature_values = X_injected_scaled[
        :,
        feature_idx
    ]

    print(
        f"Feature {feature_idx:02d}: "
        f"min={feature_values.min():.6f}, "
        f"max={feature_values.max():.6f}, "
        f"mean={feature_values.mean():.6f}, "
        f"std={feature_values.std():.6f}"
    )

# ------------------------------------------------------------
# Diagnostics specifically for injected rows
# ------------------------------------------------------------

X_injected_rows_scaled = X_injected_scaled[
    inject_idx
]

print("\nInjected-row scaled statistics:")

print(
    "Minimum:",
    float(X_injected_rows_scaled.min())
)

print(
    "Maximum:",
    float(X_injected_rows_scaled.max())
)

print(
    "Mean:",
    float(X_injected_rows_scaled.mean())
)

print(
    "Standard deviation:",
    float(X_injected_rows_scaled.std())
)

# ------------------------------------------------------------
# Save experiment metadata only
# Do not duplicate the full scaled matrix unless needed
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "inject_idx.npy",
    inject_idx
)

np.save(
    SCENARIO_RESULTS_DIR
    / "injected_ground_truth.npy",
    injected_ground_truth
)

np.savez(
    SCENARIO_RESULTS_DIR
    / "experiment_metadata.npz",
    seed=np.int64(SEED),
    scenario=np.asarray(SCENARIO),
    injection_ratio_configured=np.float64(
        INJECTION_RATIO
    ),
    injection_ratio_actual=np.float64(
        actual_injection_ratio
    ),
    total_samples=np.int64(
        n_samples
    ),
    total_features=np.int64(
        n_features
    ),
    total_injected=np.int64(
        len(inject_idx)
    ),
    injected_file=np.asarray(
        scenario_files[SCENARIO]
    )
)

print(
    "\nSaved injected experiment metadata to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected Isolation Forest evaluation
# Frozen clean model and frozen clean normalization
# ============================================================

import joblib
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

IF_BATCH_SIZE = 100_000

SELECTION_BUDGETS = {
    "0p1_percent": 0.0010,
    "0p25_percent": 0.0025,
    "0p5_percent": 0.0050,
    "1_percent": 0.0100,
    "2_percent": 0.0200
}

# ------------------------------------------------------------
# Exact top-fraction selection
# ------------------------------------------------------------

def select_exact_top_fraction(scores, fraction):
    """
    Select exactly ceil(fraction * n_samples) highest scores.

    Higher values must indicate greater anomaly likelihood.
    """

    scores = np.asarray(
        scores,
        dtype=np.float64
    ).reshape(-1)

    if scores.size == 0:
        raise ValueError("The score array is empty.")

    if not np.isfinite(scores).all():
        raise ValueError(
            "The score array contains NaN or infinite values."
        )

    if not 0 < fraction <= 1:
        raise ValueError(
            "fraction must be in the interval (0, 1]."
        )

    selected_count = int(
        np.ceil(fraction * scores.size)
    )

    selected_indices = np.argpartition(
        scores,
        scores.size - selected_count
    )[
        scores.size - selected_count:
    ]

    selected_mask = np.zeros(
        scores.size,
        dtype=np.uint8
    )

    selected_mask[selected_indices] = 1

    threshold = float(
        np.min(scores[selected_indices])
    )

    return (
        selected_mask,
        selected_indices.astype(np.int64),
        threshold
    )

# ------------------------------------------------------------
# Load frozen clean Isolation Forest artifacts
# ------------------------------------------------------------

if_model_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_IF_model.pkl"
)

if_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_IF_normalization_params.npz"
)

if not if_model_path.exists():
    raise FileNotFoundError(
        f"IF model not found: {if_model_path}"
    )

if not if_normalization_path.exists():
    raise FileNotFoundError(
        "IF normalization parameters not found: "
        f"{if_normalization_path}"
    )

if_model = joblib.load(
    if_model_path
)

if_norm_params = np.load(
    if_normalization_path
)

if_clean_p1 = float(
    if_norm_params["p1"]
)

if_clean_p99 = float(
    if_norm_params["p99"]
)

if if_clean_p99 <= if_clean_p1:
    raise ValueError(
        "Invalid saved IF normalization bounds: "
        f"p1={if_clean_p1}, p99={if_clean_p99}"
    )

print("Frozen IF model loaded.")
print(
    "Frozen clean IF normalization bounds:",
    f"p1={if_clean_p1:.8f}, "
    f"p99={if_clean_p99:.8f}"
)

# ------------------------------------------------------------
# Compute injected-dataset IF scores in batches
# ------------------------------------------------------------

def compute_if_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute IF anomaly scores in batches.

    Larger returned values indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(len(X_data) / batch_size)
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc=f"IF scoring: {SCENARIO}",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        # Same score definition as clean baseline notebook
        batch_scores = -if_model.decision_function(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


IF_scores_raw = compute_if_scores(
    model=if_model,
    X_data=X_injected_scaled,
    batch_size=IF_BATCH_SIZE
)

if IF_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected IF score shape: "
        f"{IF_scores_raw.shape}"
    )

if not np.isfinite(IF_scores_raw).all():
    raise ValueError(
        "Injected IF scores contain NaN or Inf."
    )

print(
    "Injected IF scores computed:",
    IF_scores_raw.shape
)

# ------------------------------------------------------------
# Apply frozen clean normalization
# Do not recompute p1 and p99 on injected scores
# ------------------------------------------------------------

IF_scores_clipped = np.clip(
    IF_scores_raw,
    if_clean_p1,
    if_clean_p99
)

IF_norm = (
    IF_scores_clipped - if_clean_p1
) / (
    if_clean_p99 - if_clean_p1 + 1e-8
)

IF_norm = IF_norm.astype(
    np.float32,
    copy=False
)

print(
    "Normalized IF range:",
    float(IF_norm.min()),
    "to",
    float(IF_norm.max())
)

print(
    "Normalized IF mean:",
    float(IF_norm.mean())
)

# ------------------------------------------------------------
# Save injected IF scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_IF_scores_raw.npy",
    IF_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_IF_scores_norm.npy",
    IF_norm
)

# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# ------------------------------------------------------------

if_budget_results = {}

print(
    f"\n=== IF detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=IF_norm,
        fraction=budget_fraction
    )

    selected_count = int(
        selected_mask.sum()
    )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    if_budget_results[budget_name] = {
        "budget_fraction": budget_fraction,
        "selected_count": selected_count,
        "threshold": selection_threshold,
        "detected_injected_count": detected_injected_count,
        "total_injected_count": len(inject_idx),
        "detection_rate": detection_rate,
        "maximum_possible_detection_rate":
            maximum_possible_detection_rate
    }

    # Save labels and indices for this budget
    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_anomaly_labels_{budget_name}.npy",
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_selected_indices_{budget_name}.npy",
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate={100 * detection_rate:.4f}%"
    )

# ------------------------------------------------------------
# Save compact IF budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_IF_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in if_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in if_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in if_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in if_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in if_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result["maximum_possible_detection_rate"]
            for result in if_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        if_clean_p1
    ),

    clean_normalization_p99=np.float64(
        if_clean_p99
    ),

    scenario=np.asarray(
        SCENARIO
    ),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected IF results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected LOF evaluation
# Frozen clean model and frozen clean normalization
# ============================================================

import joblib
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

LOF_BATCH_SIZE = 100_000

# Reuse:
# - SELECTION_BUDGETS
# - select_exact_top_fraction()
# - BASELINE_ARTIFACTS_DIR
# - SCENARIO_RESULTS_DIR
# - SCENARIO
# - X_injected_scaled
# - n_samples
# - inject_idx
# - actual_injection_ratio
# ------------------------------------------------------------


# ------------------------------------------------------------
# Load frozen clean LOF artifacts
# ------------------------------------------------------------

lof_model_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_LOF_model.pkl"
)

lof_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_LOF_normalization_params.npz"
)

if not lof_model_path.exists():
    raise FileNotFoundError(
        f"LOF model not found: {lof_model_path}"
    )

if not lof_normalization_path.exists():
    raise FileNotFoundError(
        "LOF normalization parameters not found: "
        f"{lof_normalization_path}"
    )

lof_model = joblib.load(
    lof_model_path
)

lof_norm_params = np.load(
    lof_normalization_path,
    allow_pickle=False
)

lof_clean_p1 = float(
    lof_norm_params["p1"]
)

lof_clean_p99 = float(
    lof_norm_params["p99"]
)

if not np.isfinite(
    [lof_clean_p1, lof_clean_p99]
).all():
    raise ValueError(
        "Saved LOF normalization bounds contain "
        "NaN or infinite values."
    )

if lof_clean_p99 <= lof_clean_p1:
    raise ValueError(
        "Invalid saved LOF normalization bounds: "
        f"p1={lof_clean_p1}, "
        f"p99={lof_clean_p99}"
    )

print("Frozen LOF model loaded.")

print(
    "LOF fitting method:",
    getattr(
        lof_model,
        "_fit_method",
        "unknown"
    )
)

print(
    "Frozen clean LOF normalization bounds:",
    f"p1={lof_clean_p1:.8f}, "
    f"p99={lof_clean_p99:.8f}"
)


# ------------------------------------------------------------
# Compute LOF raw scores for injected dataset
# ------------------------------------------------------------

def compute_lof_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute LOF anomaly scores in batches.

    score_samples() returns larger values for more normal
    observations. Negation makes larger scores more anomalous.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc=f"LOF scoring: {SCENARIO}",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        batch_scores = -model.score_samples(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    scores = np.concatenate(
        score_batches
    )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Injected LOF scores contain NaN or Inf."
        )

    return scores


LOF_scores_raw = compute_lof_scores(
    model=lof_model,
    X_data=X_injected_scaled,
    batch_size=LOF_BATCH_SIZE
)

if LOF_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected LOF score shape: "
        f"{LOF_scores_raw.shape}; "
        f"expected {(n_samples,)}"
    )

print(
    "Injected LOF scores computed:",
    LOF_scores_raw.shape
)

print("Raw minimum:", float(LOF_scores_raw.min()))
print("Raw maximum:", float(LOF_scores_raw.max()))
print("Raw mean:", float(LOF_scores_raw.mean()))


# ------------------------------------------------------------
# Apply frozen clean robust normalization
# ------------------------------------------------------------

LOF_scores_clipped = np.clip(
    LOF_scores_raw,
    lof_clean_p1,
    lof_clean_p99
)

LOF_norm = (
    LOF_scores_clipped
    - lof_clean_p1
) / (
    lof_clean_p99
    - lof_clean_p1
    + 1e-8
)

LOF_norm = LOF_norm.astype(
    np.float32,
    copy=False
)

if not np.isfinite(LOF_norm).all():
    raise ValueError(
        "Normalized LOF scores contain NaN or Inf."
    )

print("\nNormalized LOF statistics:")
print("Minimum:", float(LOF_norm.min()))
print("Maximum:", float(LOF_norm.max()))
print("Mean:", float(LOF_norm.mean()))

print(
    "Samples at normalized maximum:",
    f"{int(np.isclose(LOF_norm, 1.0).sum()):,}"
)


# ------------------------------------------------------------
# Save raw and normalized injected LOF scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_LOF_scores_raw.npy",
    LOF_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_LOF_scores_norm.npy",
    LOF_norm
)


# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# Selection uses clipped normalized LOF score
# ------------------------------------------------------------

lof_budget_results = {}

print(
    f"\n=== LOF detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in (
    SELECTION_BUDGETS.items()
):

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=LOF_norm,
        fraction=budget_fraction
    )

    selected_mask = np.asarray(
        selected_mask,
        dtype=bool
    )

    selected_indices = np.asarray(
        selected_indices,
        dtype=np.int64
    )

    selected_count = int(
        selected_mask.sum()
    )

    expected_selected_count = int(
        np.ceil(
            budget_fraction * n_samples
        )
    )

    if selected_count != expected_selected_count:
        raise ValueError(
            f"Budget {budget_name}: selected "
            f"{selected_count:,}, expected "
            f"{expected_selected_count:,}."
        )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    lof_budget_results[budget_name] = {
        "budget_fraction":
            float(budget_fraction),

        "selected_count":
            selected_count,

        "threshold":
            float(selection_threshold),

        "detected_injected_count":
            detected_injected_count,

        "total_injected_count":
            int(len(inject_idx)),

        "detection_rate":
            float(detection_rate),

        "maximum_possible_detection_rate":
            float(maximum_possible_detection_rate)
    }

    np.save(
        SCENARIO_RESULTS_DIR
        / (
            "Injected_LOF_anomaly_labels_"
            f"{budget_name}.npy"
        ),
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / (
            "Injected_LOF_selected_indices_"
            f"{budget_name}.npy"
        ),
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate="
        f"{100 * detection_rate:.4f}%"
    )


# ------------------------------------------------------------
# Save compact LOF budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_LOF_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in lof_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in lof_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in lof_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in lof_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in lof_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result[
                "maximum_possible_detection_rate"
            ]
            for result in lof_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        lof_clean_p1
    ),

    clean_normalization_p99=np.float64(
        lof_clean_p99
    ),

    selection_score_type=np.asarray(
        "clipped_clean_normalized_score"
    ),

    scoring_function=np.asarray(
        "negative_score_samples"
    ),

    scenario=np.asarray(
        SCENARIO
    ),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected LOF results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected Mahalanobis Distance evaluation
# Frozen clean parameters and frozen clean normalization
# ============================================================

import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MD_BATCH_SIZE = 100_000

# Reuse:
# - SELECTION_BUDGETS
# - select_exact_top_fraction()
# from the IF evaluation cell.

# ------------------------------------------------------------
# Load frozen clean Mahalanobis artifacts
# ------------------------------------------------------------

md_mu_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_MD_feat_mu.npy"
)

md_inv_cov_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_MD_feat_inv_cov.npy"
)

md_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_MD_normalization_params.npz"
)

required_md_paths = [
    md_mu_path,
    md_inv_cov_path,
    md_normalization_path
]

missing_md_paths = [
    path
    for path in required_md_paths
    if not path.exists()
]

if missing_md_paths:
    raise FileNotFoundError(
        "The following Mahalanobis artifacts were not found:\n"
        + "\n".join(
            str(path)
            for path in missing_md_paths
        )
    )

md_mu = np.load(
    md_mu_path
).astype(
    np.float64,
    copy=False
)

md_inv_cov = np.load(
    md_inv_cov_path
).astype(
    np.float64,
    copy=False
)

md_norm_params = np.load(
    md_normalization_path
)

md_clean_p1 = float(
    md_norm_params["p1"]
)

md_clean_p99 = float(
    md_norm_params["p99"]
)

# ------------------------------------------------------------
# Validate frozen parameters
# ------------------------------------------------------------

if md_mu.shape != (n_features,):
    raise ValueError(
        "Unexpected Mahalanobis mean shape: "
        f"{md_mu.shape}"
    )

if md_inv_cov.shape != (
    n_features,
    n_features
):
    raise ValueError(
        "Unexpected inverse covariance shape: "
        f"{md_inv_cov.shape}"
    )

if not np.isfinite(md_mu).all():
    raise ValueError(
        "Mahalanobis mean contains NaN or infinite values."
    )

if not np.isfinite(md_inv_cov).all():
    raise ValueError(
        "Mahalanobis inverse covariance contains NaN or Inf."
    )

if md_clean_p99 <= md_clean_p1:
    raise ValueError(
        "Invalid saved MD normalization bounds: "
        f"p1={md_clean_p1}, p99={md_clean_p99}"
    )

print("Frozen Mahalanobis parameters loaded.")

print(
    "Frozen clean MD normalization bounds:",
    f"p1={md_clean_p1:.8f}, "
    f"p99={md_clean_p99:.8f}"
)

# ------------------------------------------------------------
# Compute injected-dataset Mahalanobis scores
# ------------------------------------------------------------

def compute_mahalanobis_scores(
    X_data,
    mean_vector,
    inverse_covariance,
    batch_size=100_000
):
    """
    Compute Mahalanobis distances in batches.

    Larger values indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc=f"MD scoring: {SCENARIO}",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float64
        )

        diff = batch - mean_vector

        squared_distances = np.einsum(
            "ij,jk,ik->i",
            diff,
            inverse_covariance,
            diff,
            optimize=True
        )

        # Protect against tiny negative values caused by
        # floating-point precision.
        squared_distances = np.maximum(
            squared_distances,
            0.0
        )

        batch_scores = np.sqrt(
            squared_distances
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


MD_scores_raw = compute_mahalanobis_scores(
    X_data=X_injected_scaled,
    mean_vector=md_mu,
    inverse_covariance=md_inv_cov,
    batch_size=MD_BATCH_SIZE
)

if MD_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected MD score shape: "
        f"{MD_scores_raw.shape}"
    )

if not np.isfinite(MD_scores_raw).all():
    raise ValueError(
        "Injected MD scores contain NaN or infinite values."
    )

print(
    "Injected MD scores computed:",
    MD_scores_raw.shape
)

# ------------------------------------------------------------
# Apply frozen clean normalization
# Do not recompute p1 or p99 on injected scores
# ------------------------------------------------------------

MD_scores_clipped = np.clip(
    MD_scores_raw,
    md_clean_p1,
    md_clean_p99
)

MD_norm = (
    MD_scores_clipped - md_clean_p1
) / (
    md_clean_p99 - md_clean_p1 + 1e-8
)

MD_norm = MD_norm.astype(
    np.float32,
    copy=False
)

print("\nNormalized MD statistics:")
print("Minimum:", float(MD_norm.min()))
print("Maximum:", float(MD_norm.max()))
print("Mean:", float(MD_norm.mean()))

# ------------------------------------------------------------
# Save injected Mahalanobis scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_MD_scores_raw.npy",
    MD_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_MD_scores_norm.npy",
    MD_norm
)

# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# ------------------------------------------------------------

md_budget_results = {}

print(
    f"\n=== MD detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=MD_norm,
        fraction=budget_fraction
    )

    selected_count = int(
        selected_mask.sum()
    )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    md_budget_results[budget_name] = {
        "budget_fraction":
            budget_fraction,

        "selected_count":
            selected_count,

        "threshold":
            selection_threshold,

        "detected_injected_count":
            detected_injected_count,

        "total_injected_count":
            len(inject_idx),

        "detection_rate":
            detection_rate,

        "maximum_possible_detection_rate":
            maximum_possible_detection_rate
    }

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_MD_anomaly_labels_{budget_name}.npy",
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_MD_selected_indices_{budget_name}.npy",
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate={100 * detection_rate:.4f}%"
    )

# ------------------------------------------------------------
# Save compact MD budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_MD_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in md_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in md_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in md_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in md_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in md_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result["maximum_possible_detection_rate"]
            for result in md_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        md_clean_p1
    ),

    clean_normalization_p99=np.float64(
        md_clean_p99
    ),

    scenario=np.asarray(
        SCENARIO
    ),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected MD results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected IF–LOF–MD agreement analysis
# Computed separately for every selection budget
# ============================================================

agreement_summary = {}

print(
    f"=== IF–LOF–MD agreement: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    # Load masks already generated by the method cells
    if_mask = np.load(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_anomaly_labels_{budget_name}.npy"
    ).astype(bool)

    lof_mask = np.load(
        SCENARIO_RESULTS_DIR
        / f"Injected_LOF_anomaly_labels_{budget_name}.npy"
    ).astype(bool)

    md_mask = np.load(
        SCENARIO_RESULTS_DIR
        / f"Injected_MD_anomaly_labels_{budget_name}.npy"
    ).astype(bool)

    if (
        if_mask.shape != (n_samples,)
        or lof_mask.shape != (n_samples,)
        or md_mask.shape != (n_samples,)
    ):
        raise ValueError(
            f"Unexpected mask shape for budget {budget_name}."
        )

    # At least one method selects the pixel
    baseline_union = (
        if_mask
        | lof_mask
        | md_mask
    )

    # All three methods select the pixel
    baseline_intersection = (
        if_mask
        & lof_mask
        & md_mask
    )

    # Number of methods selecting each pixel:
    # 0, 1, 2, or 3
    baseline_agreement = (
        if_mask.astype(np.uint8)
        + lof_mask.astype(np.uint8)
        + md_mask.astype(np.uint8)
    )

    agreement_values, agreement_counts = np.unique(
        baseline_agreement,
        return_counts=True
    )

    union_count = int(
        baseline_union.sum()
    )

    intersection_count = int(
        baseline_intersection.sum()
    )

    union_detected = int(
        baseline_union[inject_idx].sum()
    )

    intersection_detected = int(
        baseline_intersection[inject_idx].sum()
    )

    union_detection_rate = (
        union_detected
        / len(inject_idx)
    )

    intersection_detection_rate = (
        intersection_detected
        / len(inject_idx)
    )

    agreement_summary[budget_name] = {
        "budget_fraction": budget_fraction,
        "agreement_values": agreement_values,
        "agreement_counts": agreement_counts,
        "union_count": union_count,
        "intersection_count": intersection_count,
        "union_detected": union_detected,
        "intersection_detected": intersection_detected,
        "union_detection_rate": union_detection_rate,
        "intersection_detection_rate":
            intersection_detection_rate
    }

    # Save budget-specific outputs
    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_LOF_MD_union_{budget_name}.npy",
        baseline_union.astype(np.uint8)
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_LOF_MD_intersection_{budget_name}.npy",
        baseline_intersection.astype(np.uint8)
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_IF_LOF_MD_agreement_{budget_name}.npy",
        baseline_agreement
    )

    print(
        f"\nBudget {100 * budget_fraction:.2f}%"
    )

    for value, count in zip(
        agreement_values,
        agreement_counts
    ):
        print(
            f"  Agreement={value}: "
            f"{count:,} pixels"
        )

    print(
        f"  Union: {union_count:,} pixels | "
        f"injected detected={union_detected:,}/"
        f"{len(inject_idx):,} | "
        f"DR={100 * union_detection_rate:.4f}%"
    )

    print(
        f"  Intersection: "
        f"{intersection_count:,} pixels | "
        f"injected detected={intersection_detected:,}/"
        f"{len(inject_idx):,} | "
        f"DR={100 * intersection_detection_rate:.4f}%"
    )

# Save compact summary
np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_IF_LOF_MD_agreement_summary.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        list(SELECTION_BUDGETS.values()),
        dtype=np.float64
    ),

    union_counts=np.asarray(
        [
            result["union_count"]
            for result in agreement_summary.values()
        ],
        dtype=np.int64
    ),

    intersection_counts=np.asarray(
        [
            result["intersection_count"]
            for result in agreement_summary.values()
        ],
        dtype=np.int64
    ),

    union_detected_counts=np.asarray(
        [
            result["union_detected"]
            for result in agreement_summary.values()
        ],
        dtype=np.int64
    ),

    intersection_detected_counts=np.asarray(
        [
            result["intersection_detected"]
            for result in agreement_summary.values()
        ],
        dtype=np.int64
    ),

    union_detection_rates=np.asarray(
        [
            result["union_detection_rate"]
            for result in agreement_summary.values()
        ],
        dtype=np.float64
    ),

    intersection_detection_rates=np.asarray(
        [
            result["intersection_detection_rate"]
            for result in agreement_summary.values()
        ],
        dtype=np.float64
    ),

    scenario=np.asarray(SCENARIO),
    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

In [ ]:
# ============================================================
# Injected PCA reconstruction evaluation
# Frozen clean model and frozen clean normalization
# ============================================================

import joblib
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PCA_BATCH_SIZE = 100_000

# Reuse:
# - SELECTION_BUDGETS
# - select_exact_top_fraction()
# ------------------------------------------------------------

# ------------------------------------------------------------
# Load frozen clean PCA artifacts
# ------------------------------------------------------------

pca_model_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_PCA_model.pkl"
)

pca_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_PCA_normalization_params.npz"
)

if not pca_model_path.exists():
    raise FileNotFoundError(
        f"PCA model not found: {pca_model_path}"
    )

if not pca_normalization_path.exists():
    raise FileNotFoundError(
        "PCA normalization parameters not found: "
        f"{pca_normalization_path}"
    )

pca_model = joblib.load(
    pca_model_path
)

pca_norm_params = np.load(
    pca_normalization_path
)

pca_clean_p1 = float(
    pca_norm_params["p1"]
)

pca_clean_p99 = float(
    pca_norm_params["p99"]
)

if pca_clean_p99 <= pca_clean_p1:
    raise ValueError(
        "Invalid saved PCA normalization bounds: "
        f"p1={pca_clean_p1}, "
        f"p99={pca_clean_p99}"
    )

if getattr(
    pca_model,
    "n_features_in_",
    None
) != n_features:
    raise ValueError(
        "Saved PCA model expects "
        f"{pca_model.n_features_in_} features, "
        f"but injected data has {n_features}."
    )

print("Frozen PCA model loaded.")
print(
    "Retained components:",
    pca_model.n_components_
)

print(
    "Frozen clean PCA normalization bounds:",
    f"p1={pca_clean_p1:.8f}, "
    f"p99={pca_clean_p99:.8f}"
)

# ------------------------------------------------------------
# Compute PCA reconstruction errors in batches
# ------------------------------------------------------------

def compute_pca_reconstruction_errors(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute PCA reconstruction MSE in batches.

    Larger values indicate greater anomaly likelihood.
    """

    error_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc=f"PCA scoring: {SCENARIO}",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        projected = model.transform(
            batch
        )

        reconstructed = model.inverse_transform(
            projected
        )

        batch_errors = np.mean(
            (
                batch
                - reconstructed
            ) ** 2,
            axis=1
        )

        error_batches.append(
            batch_errors.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        error_batches
    )


PCA_scores_raw = compute_pca_reconstruction_errors(
    model=pca_model,
    X_data=X_injected_scaled,
    batch_size=PCA_BATCH_SIZE
)

if PCA_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected PCA score shape: "
        f"{PCA_scores_raw.shape}"
    )

if not np.isfinite(PCA_scores_raw).all():
    raise ValueError(
        "Injected PCA scores contain NaN or Inf."
    )

print(
    "Injected PCA errors computed:",
    PCA_scores_raw.shape
)

# ------------------------------------------------------------
# Apply frozen clean normalization
# ------------------------------------------------------------

PCA_scores_clipped = np.clip(
    PCA_scores_raw,
    pca_clean_p1,
    pca_clean_p99
)

PCA_norm = (
    PCA_scores_clipped
    - pca_clean_p1
) / (
    pca_clean_p99
    - pca_clean_p1
    + 1e-8
)

PCA_norm = PCA_norm.astype(
    np.float32,
    copy=False
)

print("\nNormalized PCA statistics:")
print("Minimum:", float(PCA_norm.min()))
print("Maximum:", float(PCA_norm.max()))
print("Mean:", float(PCA_norm.mean()))
print(
    "95th percentile:",
    float(np.percentile(PCA_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(PCA_norm, 98))
)

# ------------------------------------------------------------
# Save injected PCA scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_PCA_scores_raw.npy",
    PCA_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_PCA_scores_norm.npy",
    PCA_norm
)

# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# ------------------------------------------------------------

pca_budget_results = {}

print(
    f"\n=== PCA detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=PCA_norm,
        fraction=budget_fraction
    )

    selected_count = int(
        selected_mask.sum()
    )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    pca_budget_results[budget_name] = {
        "budget_fraction":
            budget_fraction,

        "selected_count":
            selected_count,

        "threshold":
            selection_threshold,

        "detected_injected_count":
            detected_injected_count,

        "total_injected_count":
            len(inject_idx),

        "detection_rate":
            detection_rate,

        "maximum_possible_detection_rate":
            maximum_possible_detection_rate
    }

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_PCA_anomaly_labels_{budget_name}.npy",
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_PCA_selected_indices_{budget_name}.npy",
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate="
        f"{100 * detection_rate:.4f}%"
    )

# ------------------------------------------------------------
# Save compact PCA budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_PCA_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in pca_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in pca_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in pca_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in pca_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in pca_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result[
                "maximum_possible_detection_rate"
            ]
            for result in pca_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        pca_clean_p1
    ),

    clean_normalization_p99=np.float64(
        pca_clean_p99
    ),

    retained_components=np.int64(
        pca_model.n_components_
    ),

    scenario=np.asarray(SCENARIO),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected PCA results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected Autoencoder evaluation
# Frozen clean model and frozen clean normalization
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

AE_INFERENCE_BATCH_SIZE = 4_096

# Reuse:
# - SELECTION_BUDGETS
# - select_exact_top_fraction()
# - X_injected_scaled
# - n_samples, n_features
# - inject_idx
# - SCENARIO_RESULTS_DIR
# ------------------------------------------------------------

# ------------------------------------------------------------
# Define the same architecture used during clean training
# ------------------------------------------------------------

class BaselineAutoEncoder(nn.Module):

    def __init__(self, input_dimension):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dimension, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )

        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, input_dimension)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction


# ------------------------------------------------------------
# Load frozen clean AE artifacts
# ------------------------------------------------------------

ae_model_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_AE_model.pth"
)

ae_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_AE_normalization_params.npz"
)

if not ae_model_path.exists():
    raise FileNotFoundError(
        f"AE model not found: {ae_model_path}"
    )

if not ae_normalization_path.exists():
    raise FileNotFoundError(
        "AE normalization parameters not found: "
        f"{ae_normalization_path}"
    )

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    ae_model_path,
    map_location=device
)

saved_input_dim = int(
    checkpoint["input_dim"]
)

if saved_input_dim != n_features:
    raise ValueError(
        "Saved AE expects "
        f"{saved_input_dim} features, "
        f"but injected data contains {n_features}."
    )

ae_model = BaselineAutoEncoder(
    input_dimension=saved_input_dim
).to(device)

ae_model.load_state_dict(
    checkpoint["model_state_dict"]
)

ae_model.eval()

ae_norm_params = np.load(
    ae_normalization_path
)

ae_clean_p1 = float(
    ae_norm_params["p1"]
)

ae_clean_p99 = float(
    ae_norm_params["p99"]
)

if ae_clean_p99 <= ae_clean_p1:
    raise ValueError(
        "Invalid saved AE normalization bounds: "
        f"p1={ae_clean_p1}, p99={ae_clean_p99}"
    )

print("Frozen AE model loaded.")
print("AE device:", device)

print(
    "Frozen clean AE normalization bounds:",
    f"p1={ae_clean_p1:.8f}, "
    f"p99={ae_clean_p99:.8f}"
)

# ------------------------------------------------------------
# Compute reconstruction errors on injected dataset
# ------------------------------------------------------------

def compute_ae_reconstruction_errors(
    model,
    X_data,
    device,
    batch_size=4_096
):
    """
    Compute per-sample AE reconstruction MSE.

    Larger values indicate greater anomaly likelihood.
    """

    inference_tensor = torch.from_numpy(
        np.asarray(
            X_data,
            dtype=np.float32
        )
    )

    inference_loader = DataLoader(
        inference_tensor,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    error_batches = []

    model.eval()

    with torch.no_grad():

        for x_batch in tqdm(
            inference_loader,
            desc=f"AE scoring: {SCENARIO}",
            unit="batch"
        ):

            x_batch = x_batch.to(
                device,
                non_blocking=True
            )

            reconstructed = model(
                x_batch
            )

            batch_errors = torch.mean(
                (x_batch - reconstructed) ** 2,
                dim=1
            )

            error_batches.append(
                batch_errors
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False
                )
            )

    return np.concatenate(
        error_batches
    )


AE_scores_raw = compute_ae_reconstruction_errors(
    model=ae_model,
    X_data=X_injected_scaled,
    device=device,
    batch_size=AE_INFERENCE_BATCH_SIZE
)

if AE_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected AE score shape: "
        f"{AE_scores_raw.shape}"
    )

if not np.isfinite(AE_scores_raw).all():
    raise ValueError(
        "Injected AE reconstruction errors contain NaN or Inf."
    )

print(
    "Injected AE reconstruction errors computed:",
    AE_scores_raw.shape
)

# ------------------------------------------------------------
# Apply frozen clean normalization
# ------------------------------------------------------------

AE_scores_clipped = np.clip(
    AE_scores_raw,
    ae_clean_p1,
    ae_clean_p99
)

AE_norm = (
    AE_scores_clipped - ae_clean_p1
) / (
    ae_clean_p99 - ae_clean_p1 + 1e-8
)

AE_norm = AE_norm.astype(
    np.float32,
    copy=False
)

print("\nNormalized AE statistics:")
print("Minimum:", float(AE_norm.min()))
print("Maximum:", float(AE_norm.max()))
print("Mean:", float(AE_norm.mean()))
print(
    "95th percentile:",
    float(np.percentile(AE_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(AE_norm, 98))
)

# ------------------------------------------------------------
# Save injected AE scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_AE_scores_raw.npy",
    AE_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_AE_scores_norm.npy",
    AE_norm
)

# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# ------------------------------------------------------------

ae_budget_results = {}

print(
    f"\n=== AE detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=AE_norm,
        fraction=budget_fraction
    )

    selected_count = int(
        selected_mask.sum()
    )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    ae_budget_results[budget_name] = {
        "budget_fraction": budget_fraction,
        "selected_count": selected_count,
        "threshold": selection_threshold,
        "detected_injected_count": detected_injected_count,
        "total_injected_count": len(inject_idx),
        "detection_rate": detection_rate,
        "maximum_possible_detection_rate":
            maximum_possible_detection_rate
    }

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_AE_anomaly_labels_{budget_name}.npy",
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_AE_selected_indices_{budget_name}.npy",
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate={100 * detection_rate:.4f}%"
    )

# ------------------------------------------------------------
# Save compact AE budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_AE_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in ae_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in ae_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in ae_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in ae_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in ae_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result["maximum_possible_detection_rate"]
            for result in ae_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        ae_clean_p1
    ),

    clean_normalization_p99=np.float64(
        ae_clean_p99
    ),

    model_seed=np.int64(
        checkpoint["seed"]
    ),

    scenario=np.asarray(
        SCENARIO
    ),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected AE results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Injected HBOS evaluation
# Frozen clean model and frozen clean normalization
# ============================================================

import joblib
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HBOS_BATCH_SIZE = 100_000

# Reuse:
# - SELECTION_BUDGETS
# - select_exact_top_fraction()
# - X_injected_scaled
# - n_samples
# - inject_idx
# - SCENARIO
# - SCENARIO_RESULTS_DIR
# ------------------------------------------------------------

# ------------------------------------------------------------
# Load frozen clean HBOS artifacts
# ------------------------------------------------------------

hbos_model_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_HBOS_model.pkl"
)

hbos_normalization_path = (
    BASELINE_ARTIFACTS_DIR
    / "Baseline_HBOS_normalization_params.npz"
)

if not hbos_model_path.exists():
    raise FileNotFoundError(
        f"HBOS model not found: {hbos_model_path}"
    )

if not hbos_normalization_path.exists():
    raise FileNotFoundError(
        "HBOS normalization parameters not found: "
        f"{hbos_normalization_path}"
    )

hbos_model = joblib.load(
    hbos_model_path
)

hbos_norm_params = np.load(
    hbos_normalization_path
)

hbos_clean_p1 = float(
    hbos_norm_params["p1"]
)

hbos_clean_p99 = float(
    hbos_norm_params["p99"]
)

if hbos_clean_p99 <= hbos_clean_p1:
    raise ValueError(
        "Invalid saved HBOS normalization bounds: "
        f"p1={hbos_clean_p1}, p99={hbos_clean_p99}"
    )

print("Frozen HBOS model loaded.")

print(
    "Frozen clean HBOS normalization bounds:",
    f"p1={hbos_clean_p1:.8f}, "
    f"p99={hbos_clean_p99:.8f}"
)

# ------------------------------------------------------------
# Compute HBOS scores for injected dataset
# ------------------------------------------------------------

def compute_hbos_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute HBOS anomaly scores in batches.

    Larger scores indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc=f"HBOS scoring: {SCENARIO}",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        batch_scores = hbos_model.decision_function(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


HBOS_scores_raw = compute_hbos_scores(
    model=hbos_model,
    X_data=X_injected_scaled,
    batch_size=HBOS_BATCH_SIZE
)

if HBOS_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected HBOS score shape: "
        f"{HBOS_scores_raw.shape}"
    )

if not np.isfinite(HBOS_scores_raw).all():
    raise ValueError(
        "Injected HBOS scores contain NaN or infinite values."
    )

print(
    "Injected HBOS scores computed:",
    HBOS_scores_raw.shape
)

# ------------------------------------------------------------
# Apply frozen clean normalization
# ------------------------------------------------------------

HBOS_scores_clipped = np.clip(
    HBOS_scores_raw,
    hbos_clean_p1,
    hbos_clean_p99
)

HBOS_norm = (
    HBOS_scores_clipped - hbos_clean_p1
) / (
    hbos_clean_p99 - hbos_clean_p1 + 1e-8
)

HBOS_norm = HBOS_norm.astype(
    np.float32,
    copy=False
)

print("\nNormalized HBOS statistics:")
print("Minimum:", float(HBOS_norm.min()))
print("Maximum:", float(HBOS_norm.max()))
print("Mean:", float(HBOS_norm.mean()))
print(
    "95th percentile:",
    float(np.percentile(HBOS_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(HBOS_norm, 98))
)

# ------------------------------------------------------------
# Save injected HBOS scores
# ------------------------------------------------------------

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_HBOS_scores_raw.npy",
    HBOS_scores_raw
)

np.save(
    SCENARIO_RESULTS_DIR
    / "Injected_HBOS_scores_norm.npy",
    HBOS_norm
)

# ------------------------------------------------------------
# Evaluate all fixed selection budgets
# ------------------------------------------------------------

hbos_budget_results = {}

print(
    f"\n=== HBOS detection results: {SCENARIO} ==="
)

for budget_name, budget_fraction in SELECTION_BUDGETS.items():

    (
        selected_mask,
        selected_indices,
        selection_threshold
    ) = select_exact_top_fraction(
        scores=HBOS_norm,
        fraction=budget_fraction
    )

    selected_count = int(
        selected_mask.sum()
    )

    detected_injected_count = int(
        selected_mask[inject_idx].sum()
    )

    detection_rate = (
        detected_injected_count
        / len(inject_idx)
    )

    maximum_possible_detection_rate = min(
        1.0,
        selected_count / len(inject_idx)
    )

    hbos_budget_results[budget_name] = {
        "budget_fraction":
            budget_fraction,

        "selected_count":
            selected_count,

        "threshold":
            selection_threshold,

        "detected_injected_count":
            detected_injected_count,

        "total_injected_count":
            len(inject_idx),

        "detection_rate":
            detection_rate,

        "maximum_possible_detection_rate":
            maximum_possible_detection_rate
    }

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_HBOS_anomaly_labels_{budget_name}.npy",
        selected_mask
    )

    np.save(
        SCENARIO_RESULTS_DIR
        / f"Injected_HBOS_selected_indices_{budget_name}.npy",
        selected_indices
    )

    print(
        f"Budget {100 * budget_fraction:.2f}% | "
        f"selected={selected_count:,} | "
        f"detected={detected_injected_count:,}/"
        f"{len(inject_idx):,} | "
        f"detection rate={100 * detection_rate:.4f}%"
    )

# ------------------------------------------------------------
# Save compact HBOS budget summary
# ------------------------------------------------------------

np.savez(
    SCENARIO_RESULTS_DIR
    / "Injected_HBOS_budget_results.npz",

    budget_names=np.asarray(
        list(SELECTION_BUDGETS.keys())
    ),

    budget_fractions=np.asarray(
        [
            result["budget_fraction"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.float64
    ),

    selected_counts=np.asarray(
        [
            result["selected_count"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.int64
    ),

    thresholds=np.asarray(
        [
            result["threshold"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.float64
    ),

    detected_injected_counts=np.asarray(
        [
            result["detected_injected_count"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.int64
    ),

    total_injected_count=np.int64(
        len(inject_idx)
    ),

    detection_rates=np.asarray(
        [
            result["detection_rate"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.float64
    ),

    maximum_possible_detection_rates=np.asarray(
        [
            result["maximum_possible_detection_rate"]
            for result in hbos_budget_results.values()
        ],
        dtype=np.float64
    ),

    clean_normalization_p1=np.float64(
        hbos_clean_p1
    ),

    clean_normalization_p99=np.float64(
        hbos_clean_p99
    ),

    scenario=np.asarray(
        SCENARIO
    ),

    injection_ratio=np.float64(
        actual_injection_ratio
    )
)

print(
    "\nInjected HBOS results saved to:",
    SCENARIO_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Final injected-baseline summary and Excel export
# Uses openpyxl
# ============================================================

from pathlib import Path

import numpy as np
from openpyxl import Workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

METHODS = [
    "IF",
    "LOF",
    "MD",
    "PCA",
    "AE",
    "HBOS"
]

BUDGET_ORDER = [
    "0p1_percent",
    "0p25_percent",
    "0p5_percent",
    "1_percent",
    "2_percent"
]

BUDGET_LABELS = {
    "0p1_percent": "0.10%",
    "0p25_percent": "0.25%",
    "0p5_percent": "0.50%",
    "1_percent": "1.00%",
    "2_percent": "2.00%"
}

# ------------------------------------------------------------
# Validate injection information
# ------------------------------------------------------------

if inject_idx.ndim != 1:
    raise ValueError(
        "inject_idx must be one-dimensional."
    )

if len(inject_idx) == 0:
    raise ValueError(
        "No injected indices were loaded."
    )

if np.unique(inject_idx).size != len(inject_idx):
    raise ValueError(
        "inject_idx contains duplicate indices."
    )

print("Scenario:", SCENARIO)
print("Injected samples:", f"{len(inject_idx):,}")
print("Total samples:", f"{n_samples:,}")
print(
    "Actual injection rate:",
    f"{100 * actual_injection_ratio:.6f}%"
)

# ------------------------------------------------------------
# Load and validate method result summaries
# ------------------------------------------------------------

result_rows = []
method_detection_rates = {}

for method_name in METHODS:

    result_path = (
        SCENARIO_RESULTS_DIR
        / f"Injected_{method_name}_budget_results.npz"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Missing result file: {result_path}"
        )

    result_data = np.load(
        result_path,
        allow_pickle=False
    )

    budget_names = [
        str(value)
        for value in result_data["budget_names"]
    ]

    budget_fractions = np.asarray(
        result_data["budget_fractions"],
        dtype=np.float64
    )

    selected_counts = np.asarray(
        result_data["selected_counts"],
        dtype=np.int64
    )

    thresholds = np.asarray(
        result_data["thresholds"],
        dtype=np.float64
    )

    detected_counts = np.asarray(
        result_data["detected_injected_counts"],
        dtype=np.int64
    )

    detection_rates = np.asarray(
        result_data["detection_rates"],
        dtype=np.float64
    )

    maximum_rates = np.asarray(
        result_data[
            "maximum_possible_detection_rates"
        ],
        dtype=np.float64
    )

    total_injected_saved = int(
        result_data["total_injected_count"]
    )

    if total_injected_saved != len(inject_idx):
        raise ValueError(
            f"{method_name}: saved injected count "
            f"{total_injected_saved:,} does not match "
            f"current count {len(inject_idx):,}."
        )

    if budget_names != BUDGET_ORDER:
        raise ValueError(
            f"{method_name}: unexpected budget order: "
            f"{budget_names}"
        )

    method_detection_rates[
        method_name
    ] = detection_rates

    for position, budget_name in enumerate(
        budget_names
    ):

        expected_selected_count = int(
            np.ceil(
                budget_fractions[position]
                * n_samples
            )
        )

        if (
            selected_counts[position]
            != expected_selected_count
        ):
            raise ValueError(
                f"{method_name}, {budget_name}: "
                f"selected {selected_counts[position]:,}, "
                f"expected {expected_selected_count:,}."
            )

        independently_computed_rate = (
            detected_counts[position]
            / len(inject_idx)
        )

        if not np.isclose(
            detection_rates[position],
            independently_computed_rate
        ):
            raise ValueError(
                f"{method_name}, {budget_name}: "
                "saved detection rate is inconsistent."
            )

        result_rows.append([
            SCENARIO,
            float(actual_injection_ratio),
            method_name,
            BUDGET_LABELS[budget_name],
            float(budget_fractions[position]),
            int(selected_counts[position]),
            int(detected_counts[position]),
            int(len(inject_idx)),
            float(detection_rates[position]),
            float(maximum_rates[position]),
            float(thresholds[position])
        ])

# ------------------------------------------------------------
# Print final result table
# ------------------------------------------------------------

print(
    f"\n=== Injected baseline results: "
    f"{SCENARIO} ==="
)

print(
    f"{'Method':<8}"
    f"{'Budget':>10}"
    f"{'Selected':>14}"
    f"{'Detected':>14}"
    f"{'DR (%)':>12}"
)

print("-" * 58)

for row in result_rows:

    print(
        f"{row[2]:<8}"
        f"{row[3]:>10}"
        f"{row[5]:>14,}"
        f"{row[6]:>14,}"
        f"{100 * row[8]:>11.4f}"
    )

# ------------------------------------------------------------
# Create Excel workbook
# ------------------------------------------------------------

workbook = Workbook()

# Remove default worksheet and recreate named sheets
default_sheet = workbook.active
workbook.remove(default_sheet)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)

header_font = Font(
    bold=True,
    color="FFFFFF"
)

center_alignment = Alignment(
    horizontal="center",
    vertical="center"
)

# ============================================================
# Sheet 1: Detailed Results
# ============================================================

detail_sheet = workbook.create_sheet(
    "Detailed Results"
)

detail_headers = [
    "Scenario",
    "Injection Rate",
    "Method",
    "Selection Budget",
    "Budget Fraction",
    "Selected Count",
    "Detected Injected",
    "Total Injected",
    "Detection Rate",
    "Maximum Possible DR",
    "Selection Threshold"
]

detail_sheet.append(
    detail_headers
)

for row in result_rows:
    detail_sheet.append(row)

for cell in detail_sheet[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = center_alignment

detail_sheet.freeze_panes = "A2"
detail_sheet.auto_filter.ref = (
    f"A1:K{detail_sheet.max_row}"
)

for row_idx in range(
    2,
    detail_sheet.max_row + 1
):
    detail_sheet.cell(
        row=row_idx,
        column=2
    ).number_format = "0.0000%"

    detail_sheet.cell(
        row=row_idx,
        column=5
    ).number_format = "0.00%"

    detail_sheet.cell(
        row=row_idx,
        column=9
    ).number_format = "0.00%"

    detail_sheet.cell(
        row=row_idx,
        column=10
    ).number_format = "0.00%"

    detail_sheet.cell(
        row=row_idx,
        column=11
    ).number_format = "0.000000"

detail_widths = {
    "A": 18,
    "B": 16,
    "C": 12,
    "D": 18,
    "E": 16,
    "F": 16,
    "G": 18,
    "H": 16,
    "I": 16,
    "J": 21,
    "K": 21
}

for column, width in detail_widths.items():
    detail_sheet.column_dimensions[
        column
    ].width = width

detail_table = Table(
    displayName="InjectedBaselineDetailedResults",
    ref=f"A1:K{detail_sheet.max_row}"
)

detail_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

detail_sheet.add_table(
    detail_table
)

# ============================================================
# Sheet 2: Detection Rate Matrix
# ============================================================

matrix_sheet = workbook.create_sheet(
    "Detection Rate Matrix"
)

matrix_headers = [
    "Method",
    "0.10%",
    "0.25%",
    "0.50%",
    "1.00%",
    "2.00%"
]

matrix_sheet.append(
    matrix_headers
)

for method_name in METHODS:

    matrix_sheet.append([
        method_name,
        *[
            float(value)
            for value in method_detection_rates[
                method_name
            ]
        ]
    ])

for cell in matrix_sheet[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = center_alignment

matrix_sheet.freeze_panes = "A2"

for row_idx in range(
    2,
    matrix_sheet.max_row + 1
):
    for column_idx in range(
        2,
        7
    ):
        matrix_sheet.cell(
            row=row_idx,
            column=column_idx
        ).number_format = "0.00%"

matrix_sheet.column_dimensions["A"].width = 14

for column_idx in range(
    2,
    7
):
    matrix_sheet.column_dimensions[
        get_column_letter(column_idx)
    ].width = 13

matrix_sheet.conditional_formatting.add(
    f"B2:F{matrix_sheet.max_row}",
    ColorScaleRule(
        start_type="min",
        start_color="FEE2E2",
        mid_type="percentile",
        mid_value=50,
        mid_color="FEF3C7",
        end_type="max",
        end_color="DCFCE7"
    )
)

matrix_table = Table(
    displayName="InjectedBaselineDetectionMatrix",
    ref=f"A1:F{matrix_sheet.max_row}"
)

matrix_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

matrix_sheet.add_table(
    matrix_table
)

# ============================================================
# Sheet 3: Experiment Metadata
# ============================================================

metadata_sheet = workbook.create_sheet(
    "Experiment Metadata"
)

metadata_rows = [
    ["Parameter", "Value"],
    ["Seed", int(SEED)],
    ["Scenario", SCENARIO],
    [
        "Configured injection rate",
        float(INJECTION_RATIO)
    ],
    [
        "Actual injection rate",
        float(actual_injection_ratio)
    ],
    ["Total samples", int(n_samples)],
    ["Total features", int(n_features)],
    ["Injected samples", int(len(inject_idx))],
    [
        "Injected dataset file",
        scenario_files[SCENARIO]
    ],
    [
        "Injected dataset folder",
        str(INJECTED_DATA_DIR)
    ],
    [
        "Baseline artifact folder",
        str(BASELINE_ARTIFACTS_DIR)
    ],
    [
        "Scenario result folder",
        str(SCENARIO_RESULTS_DIR)
    ],
    [
        "Evaluation note",
        (
            "Detection rate is calculated only on known "
            "synthetically injected indices. Non-injected "
            "pixels are not treated as confirmed negatives."
        )
    ]
]

for row in metadata_rows:
    metadata_sheet.append(row)

for cell in metadata_sheet[1]:
    cell.fill = header_fill
    cell.font = header_font

metadata_sheet.column_dimensions["A"].width = 31
metadata_sheet.column_dimensions["B"].width = 75

for row_idx in range(
    2,
    metadata_sheet.max_row + 1
):
    metadata_sheet.cell(
        row=row_idx,
        column=2
    ).alignment = Alignment(
        wrap_text=True,
        vertical="top"
    )

metadata_sheet["B4"].number_format = "0.0000%"
metadata_sheet["B5"].number_format = "0.0000%"

# ------------------------------------------------------------
# Save Excel workbook
# ------------------------------------------------------------

excel_path = (
    SCENARIO_RESULTS_DIR
    / (
        f"Injected_Baseline_Results_"
        f"{SCENARIO}_"
        f"{ratio_tag}.xlsx"
    )
)

workbook.save(
    excel_path
)

print(
    "\nExcel results saved to:",
    excel_path.resolve()
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Main paths
# ============================================================

BASE_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

BASELINE_RESULTS_DIR = (
    BASE_DIR
    / "02_Injected_Baseline_Results_Seed_42"
    / "ratio_0p00050"
)

INJECTED_DATA_DIR = (
    BASE_DIR
    / "03_Injected_Proposed_Results_Seed_42"
    / "Injected_Datasets_with_0.05%_Injection_Rate"
)

INJECTED_INDEX_PATH = INJECTED_DATA_DIR / "inject_indices.npy"

OUTPUT_CSV = (
    BASELINE_RESULTS_DIR
    / "Injected_Baseline_Detection_Rates_Raw_Scores.csv"
)


# ============================================================
# 2. Scenarios, methods, and evaluation budgets
# ============================================================

SCENARIOS = [
    "extreme",
    "inconsistent",
    "noise_mild",
    "noise_moderate",
    "noise_strong",
]

METHOD_FILES = {
    "IF": "Injected_IF_scores_raw.npy",
    "LOF": "Injected_LOF_scores_raw.npy",
    "MD": "Injected_MD_scores_raw.npy",
    "PCA": "Injected_PCA_scores_raw.npy",
    "AE": "Injected_AE_scores_raw.npy",
    "HBOS": "Injected_HBOS_scores_raw.npy",
}

# Fractions of the complete pixel dataset selected as anomalous
BUDGETS = [
    0.0010,   # 0.10%
    0.0025,   # 0.25%
    0.0050,   # 0.50%
    0.0100,   # 1.00%
    0.0200,   # 2.00%
]


# ============================================================
# 3. Load and validate injected indices
# ============================================================

if not INJECTED_INDEX_PATH.exists():
    raise FileNotFoundError(
        f"Injected-index file was not found:\n"
        f"{INJECTED_INDEX_PATH}"
    )

inject_idx = np.load(INJECTED_INDEX_PATH).astype(
    np.int64,
    copy=False,
)

if inject_idx.ndim != 1:
    raise ValueError(
        "inject_indices.npy must contain a one-dimensional array."
    )

if inject_idx.size == 0:
    raise ValueError(
        "inject_indices.npy does not contain any indices."
    )

if np.unique(inject_idx).size != inject_idx.size:
    raise ValueError(
        "inject_indices.npy contains duplicate indices."
    )

print("Injected-index file:", INJECTED_INDEX_PATH)
print("Number of injected samples:", f"{inject_idx.size:,}")


# ============================================================
# 4. Exact top-k selection
# ============================================================

def exact_top_k_indices(
    scores: np.ndarray,
    k: int,
) -> np.ndarray:
    """
    Return exactly k indices having the largest anomaly scores.

    Larger scores must indicate greater anomaly likelihood.
    np.argpartition avoids sorting the complete 4.29-million-element
    array for every budget.
    """
    scores = np.asarray(scores).reshape(-1)

    if not np.issubdtype(scores.dtype, np.number):
        raise TypeError("The score array must be numeric.")

    if not np.all(np.isfinite(scores)):
        bad_count = int((~np.isfinite(scores)).sum())
        raise ValueError(
            f"The score array contains {bad_count:,} "
            "NaN or infinite values."
        )

    n_samples = scores.size

    if not 1 <= k <= n_samples:
        raise ValueError(
            f"k must be between 1 and {n_samples:,}; received {k:,}."
        )

    # Select the k largest values without fully sorting all samples.
    top_idx = np.argpartition(scores, n_samples - k)[-k:]

    # Optional deterministic ordering within the selected top-k set.
    # Higher score first; index is used to break exact score ties.
    order = np.lexsort(
        (
            top_idx,
            -scores[top_idx],
        )
    )

    return top_idx[order]


# ============================================================
# 5. Detection-rate calculation
# ============================================================

def evaluate_detection_rates(
    scores: np.ndarray,
    injected_indices: np.ndarray,
    budgets: list[float],
) -> list[dict]:
    """
    Calculate injected-anomaly detection rate at each top-k budget.

    Detection rate:
        detected injected samples / total injected samples * 100
    """
    scores = np.asarray(scores).reshape(-1)
    n_samples = scores.size

    if injected_indices.min() < 0:
        raise ValueError("Injected indices cannot be negative.")

    if injected_indices.max() >= n_samples:
        raise IndexError(
            "At least one injected index exceeds the score-array length."
        )

    results = []

    for budget in budgets:
        # Matches the previous notebook convention:
        # ceil(q * number of samples)
        selected_count = int(np.ceil(budget * n_samples))

        selected_idx = exact_top_k_indices(
            scores=scores,
            k=selected_count,
        )

        # Count the intersection between injected and selected indices.
        detected_count = int(
            np.isin(
                injected_indices,
                selected_idx,
                assume_unique=True,
            ).sum()
        )

        missed_count = int(
            injected_indices.size - detected_count
        )

        detection_rate = (
            100.0
            * detected_count
            / injected_indices.size
        )

        miss_rate = (
            100.0
            * missed_count
            / injected_indices.size
        )

        results.append(
            {
                "Budget_Fraction": budget,
                "Budget_Percent": 100.0 * budget,
                "Selected_Count": selected_count,
                "Detected_Injected": detected_count,
                "Missed_Injected": missed_count,
                "Total_Injected": int(injected_indices.size),
                "Detection_Rate_Percent": detection_rate,
                "Miss_Rate_Percent": miss_rate,
            }
        )

    return results


# ============================================================
# 6. Evaluate every scenario and baseline
# ============================================================

all_results = []

for scenario in SCENARIOS:
    scenario_dir = BASELINE_RESULTS_DIR / scenario

    if not scenario_dir.exists():
        raise FileNotFoundError(
            f"Scenario folder was not found:\n{scenario_dir}"
        )

    print("\n" + "=" * 72)
    print("Scenario:", scenario)
    print("Folder:", scenario_dir)
    print("=" * 72)

    expected_n_samples = None

    for method, filename in METHOD_FILES.items():
        score_path = scenario_dir / filename

        if not score_path.exists():
            raise FileNotFoundError(
                f"Missing raw-score file for {method}:\n"
                f"{score_path}"
            )

        # Memory mapping avoids unnecessarily copying the full array.
        scores = np.load(
            score_path,
            mmap_mode="r",
        )

        scores = np.asarray(scores).reshape(-1)

        if expected_n_samples is None:
            expected_n_samples = scores.size
        elif scores.size != expected_n_samples:
            raise ValueError(
                f"Score length mismatch in scenario '{scenario}'. "
                f"{method} contains {scores.size:,} scores, while "
                f"the first method contains "
                f"{expected_n_samples:,} scores."
            )

        print(
            f"\n{method}: loaded {scores.size:,} raw scores"
        )
        print(
            f"Raw range: "
            f"{float(np.min(scores)):.8f} to "
            f"{float(np.max(scores)):.8f}"
        )

        method_results = evaluate_detection_rates(
            scores=scores,
            injected_indices=inject_idx,
            budgets=BUDGETS,
        )

        for result in method_results:
            result["Scenario"] = scenario
            result["Method"] = method
            result["Score_File"] = filename

            all_results.append(result)

            print(
                f"Budget {result['Budget_Percent']:.2f}% | "
                f"selected={result['Selected_Count']:,} | "
                f"detected="
                f"{result['Detected_Injected']:,}/"
                f"{result['Total_Injected']:,} | "
                f"detection rate="
                f"{result['Detection_Rate_Percent']:.4f}%"
            )


# ============================================================
# 7. Create and save result table
# ============================================================

results_df = pd.DataFrame(all_results)

column_order = [
    "Scenario",
    "Method",
    "Budget_Fraction",
    "Budget_Percent",
    "Selected_Count",
    "Detected_Injected",
    "Missed_Injected",
    "Total_Injected",
    "Detection_Rate_Percent",
    "Miss_Rate_Percent",
    "Score_File",
]

results_df = results_df[column_order]

results_df = results_df.sort_values(
    by=[
        "Scenario",
        "Method",
        "Budget_Fraction",
    ],
    ignore_index=True,
)

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\n" + "=" * 72)
print("RAW-SCORE BASELINE EVALUATION COMPLETED")
print("=" * 72)
print("Results saved to:")
print(OUTPUT_CSV)

print("\nTop-2% results:")
print(
    results_df.loc[
        np.isclose(
            results_df["Budget_Fraction"],
            0.02,
        ),
        [
            "Scenario",
            "Method",
            "Detected_Injected",
            "Total_Injected",
            "Detection_Rate_Percent",
        ],
    ].to_string(index=False)
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Main paths
# ============================================================

BASE_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

BASELINE_RESULTS_DIR = (
    BASE_DIR
    / "02_Injected_Baseline_Results_Seed_42"
    / "ratio_0p00050"
)

INJECTED_DATA_DIR = (
    BASE_DIR
    / "03_Injected_Proposed_Results_Seed_42"
    / "Injected_Datasets_with_0.05%_Injection_Rate"
)

INJECTED_INDEX_PATH = (
    INJECTED_DATA_DIR
    / "inject_indices.npy"
)

OUTPUT_CSV = (
    BASELINE_RESULTS_DIR
    / "Injected_Baseline_Detection_Rates_Raw_Scores.csv"
)

OUTPUT_EXCEL = (
    BASELINE_RESULTS_DIR
    / "Injected_Baseline_Detection_Rates_Raw_Scores.xlsx"
)


# ============================================================
# 2. Scenarios, methods, and evaluation budgets
# ============================================================

SCENARIOS = [
    "extreme",
    "inconsistent",
    "noise_mild",
    "noise_moderate",
    "noise_strong",
]

METHOD_FILES = {
    "IF": "Injected_IF_scores_raw.npy",
    "LOF": "Injected_LOF_scores_raw.npy",
    "MD": "Injected_MD_scores_raw.npy",
    "PCA": "Injected_PCA_scores_raw.npy",
    "AE": "Injected_AE_scores_raw.npy",
    "HBOS": "Injected_HBOS_scores_raw.npy",
}

# Fractions of the complete dataset selected as anomalous
BUDGETS = [
    0.0010,   # 0.10%
    0.0025,   # 0.25%
    0.0050,   # 0.50%
    0.0100,   # 1.00%
    0.0200,   # 2.00%
]


# ============================================================
# 3. Load and validate injected indices
# ============================================================

if not INJECTED_INDEX_PATH.exists():
    raise FileNotFoundError(
        "Injected-index file was not found:\n"
        f"{INJECTED_INDEX_PATH}"
    )

inject_idx = np.load(
    INJECTED_INDEX_PATH
).astype(
    np.int64,
    copy=False,
)

inject_idx = np.asarray(
    inject_idx
).reshape(-1)

if inject_idx.size == 0:
    raise ValueError(
        "inject_indices.npy does not contain any indices."
    )

if np.any(inject_idx < 0):
    raise ValueError(
        "inject_indices.npy contains negative indices."
    )

if np.unique(inject_idx).size != inject_idx.size:
    raise ValueError(
        "inject_indices.npy contains duplicate indices."
    )

print("Injected-index file:")
print(INJECTED_INDEX_PATH)
print(
    "Number of injected samples:",
    f"{inject_idx.size:,}"
)


# ============================================================
# 4. Exact top-k selection
# ============================================================

def exact_top_k_indices(
    scores: np.ndarray,
    k: int,
) -> np.ndarray:
    """
    Return exactly k indices with the largest anomaly scores.

    Assumption:
        Larger score = more anomalous.
    """

    scores = np.asarray(
        scores
    ).reshape(-1)

    if not np.issubdtype(
        scores.dtype,
        np.number,
    ):
        raise TypeError(
            "The score array must be numeric."
        )

    finite_mask = np.isfinite(scores)

    if not np.all(finite_mask):
        bad_count = int(
            (~finite_mask).sum()
        )

        raise ValueError(
            f"The score array contains {bad_count:,} "
            "NaN or infinite values."
        )

    n_samples = scores.size

    if not 1 <= k <= n_samples:
        raise ValueError(
            f"k must be between 1 and {n_samples:,}. "
            f"Received k={k:,}."
        )

    # Select exactly k largest score positions.
    top_idx = np.argpartition(
        scores,
        n_samples - k,
    )[-k:]

    # Sort selected samples by descending score.
    # The sample index acts as a deterministic tie-breaker.
    order = np.lexsort(
        (
            top_idx,
            -scores[top_idx],
        )
    )

    return top_idx[order]


# ============================================================
# 5. Detection-rate calculation
# ============================================================

def evaluate_detection_rates(
    scores: np.ndarray,
    injected_indices: np.ndarray,
    budgets: list[float],
) -> list[dict]:
    """
    Calculate injected-anomaly detection rate at each budget.

    Detection rate:
        detected injected samples
        ------------------------- x 100
        total injected samples
    """

    scores = np.asarray(
        scores
    ).reshape(-1)

    n_samples = scores.size

    if injected_indices.max() >= n_samples:
        raise IndexError(
            "At least one injected index exceeds the "
            "score-array length."
        )

    method_results = []

    for budget in budgets:

        selected_count = int(
            np.ceil(
                budget * n_samples
            )
        )

        selected_idx = exact_top_k_indices(
            scores=scores,
            k=selected_count,
        )

        # Both arrays contain unique indices.
        detected_count = int(
            np.intersect1d(
                injected_indices,
                selected_idx,
                assume_unique=True,
            ).size
        )

        missed_count = int(
            injected_indices.size
            - detected_count
        )

        detection_rate = (
            detected_count
            / injected_indices.size
            * 100.0
        )

        miss_rate = (
            missed_count
            / injected_indices.size
            * 100.0
        )

        method_results.append(
            {
                "Budget_Fraction": budget,
                "Budget_Percent": budget * 100.0,
                "Selected_Count": selected_count,
                "Detected_Injected": detected_count,
                "Missed_Injected": missed_count,
                "Total_Injected": int(
                    injected_indices.size
                ),
                "Detection_Rate_Percent": detection_rate,
                "Miss_Rate_Percent": miss_rate,
            }
        )

    return method_results


# ============================================================
# 6. Evaluate all scenarios and baseline methods
# ============================================================

all_results = []
score_summary = []

for scenario in SCENARIOS:

    scenario_dir = (
        BASELINE_RESULTS_DIR
        / scenario
    )

    if not scenario_dir.exists():
        raise FileNotFoundError(
            "Scenario folder was not found:\n"
            f"{scenario_dir}"
        )

    print("\n" + "=" * 75)
    print("Scenario:", scenario)
    print("Folder:", scenario_dir)
    print("=" * 75)

    expected_n_samples = None

    for method, filename in METHOD_FILES.items():

        score_path = (
            scenario_dir
            / filename
        )

        if not score_path.exists():
            raise FileNotFoundError(
                f"Missing raw-score file for {method}:\n"
                f"{score_path}"
            )

        # Memory map the score array to reduce memory usage.
        scores = np.load(
            score_path,
            mmap_mode="r",
        )

        scores = np.asarray(
            scores
        ).reshape(-1)

        if expected_n_samples is None:
            expected_n_samples = scores.size

        elif scores.size != expected_n_samples:
            raise ValueError(
                f"Score length mismatch in scenario "
                f"'{scenario}'.\n"
                f"{method}: {scores.size:,}\n"
                f"Expected: {expected_n_samples:,}"
            )

        score_min = float(
            np.min(scores)
        )

        score_max = float(
            np.max(scores)
        )

        score_mean = float(
            np.mean(scores)
        )

        score_std = float(
            np.std(scores)
        )

        print(
            f"\n{method}: "
            f"loaded {scores.size:,} raw scores"
        )

        print(
            f"Raw score range: "
            f"{score_min:.8f} to "
            f"{score_max:.8f}"
        )

        score_summary.append(
            {
                "Scenario": scenario,
                "Method": method,
                "Number_of_Scores": scores.size,
                "Score_Minimum": score_min,
                "Score_Maximum": score_max,
                "Score_Mean": score_mean,
                "Score_Std": score_std,
                "Score_File": filename,
                "Score_Path": str(score_path),
            }
        )

        method_results = evaluate_detection_rates(
            scores=scores,
            injected_indices=inject_idx,
            budgets=BUDGETS,
        )

        for result in method_results:

            result["Scenario"] = scenario
            result["Method"] = method
            result["Number_of_Samples"] = scores.size
            result["Injection_Ratio_Percent"] = (
                inject_idx.size
                / scores.size
                * 100.0
            )
            result["Score_File"] = filename

            all_results.append(result)

            print(
                f"Budget "
                f"{result['Budget_Percent']:.2f}% | "
                f"selected="
                f"{result['Selected_Count']:,} | "
                f"detected="
                f"{result['Detected_Injected']:,}/"
                f"{result['Total_Injected']:,} | "
                f"detection rate="
                f"{result['Detection_Rate_Percent']:.4f}%"
            )


# ============================================================
# 7. Create the detailed results DataFrame
# ============================================================

results_df = pd.DataFrame(
    all_results
)

column_order = [
    "Scenario",
    "Method",
    "Budget_Fraction",
    "Budget_Percent",
    "Selected_Count",
    "Detected_Injected",
    "Missed_Injected",
    "Total_Injected",
    "Detection_Rate_Percent",
    "Miss_Rate_Percent",
    "Number_of_Samples",
    "Injection_Ratio_Percent",
    "Score_File",
]

results_df = results_df[
    column_order
]

results_df = results_df.sort_values(
    by=[
        "Scenario",
        "Budget_Fraction",
        "Method",
    ],
    ignore_index=True,
)

score_summary_df = pd.DataFrame(
    score_summary
)

score_summary_df = score_summary_df.sort_values(
    by=[
        "Scenario",
        "Method",
    ],
    ignore_index=True,
)


# ============================================================
# 8. Create summary tables for Excel
# ============================================================

# Detection rate table:
# rows = scenario and method
# columns = anomaly-selection budgets
detection_rate_pivot = results_df.pivot_table(
    index=[
        "Scenario",
        "Method",
    ],
    columns="Budget_Percent",
    values="Detection_Rate_Percent",
    aggfunc="first",
)

detection_rate_pivot = (
    detection_rate_pivot
    .reset_index()
)

detection_rate_pivot.columns.name = None

detection_rate_pivot = (
    detection_rate_pivot.rename(
        columns={
            0.10: "DR_at_0.10_Percent",
            0.25: "DR_at_0.25_Percent",
            0.50: "DR_at_0.50_Percent",
            1.00: "DR_at_1.00_Percent",
            2.00: "DR_at_2.00_Percent",
        }
    )
)


# Detected-count table
detected_count_pivot = results_df.pivot_table(
    index=[
        "Scenario",
        "Method",
    ],
    columns="Budget_Percent",
    values="Detected_Injected",
    aggfunc="first",
)

detected_count_pivot = (
    detected_count_pivot
    .reset_index()
)

detected_count_pivot.columns.name = None

detected_count_pivot = (
    detected_count_pivot.rename(
        columns={
            0.10: "Detected_at_0.10_Percent",
            0.25: "Detected_at_0.25_Percent",
            0.50: "Detected_at_0.50_Percent",
            1.00: "Detected_at_1.00_Percent",
            2.00: "Detected_at_2.00_Percent",
        }
    )
)


# Mean detection rate across all five scenarios
average_detection_df = (
    results_df
    .groupby(
        [
            "Method",
            "Budget_Fraction",
            "Budget_Percent",
        ],
        as_index=False,
    )
    .agg(
        Mean_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "mean",
        ),
        Std_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "std",
        ),
        Minimum_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "min",
        ),
        Maximum_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "max",
        ),
    )
)

average_detection_df = (
    average_detection_df
    .sort_values(
        by=[
            "Budget_Fraction",
            "Mean_Detection_Rate_Percent",
        ],
        ascending=[
            True,
            False,
        ],
        ignore_index=True,
    )
)


# Ranking of methods for each scenario and budget
ranking_df = results_df.copy()

ranking_df[
    "Detection_Rate_Rank"
] = (
    ranking_df
    .groupby(
        [
            "Scenario",
            "Budget_Fraction",
        ]
    )[
        "Detection_Rate_Percent"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

ranking_df = ranking_df.sort_values(
    by=[
        "Scenario",
        "Budget_Fraction",
        "Detection_Rate_Rank",
        "Method",
    ],
    ignore_index=True,
)


# Experimental metadata
metadata_df = pd.DataFrame(
    {
        "Setting": [
            "Baseline result folder",
            "Injected index file",
            "Injection ratio folder",
            "Number of injected samples",
            "Scenarios",
            "Methods",
            "Evaluation budgets",
            "Score selection",
            "Score direction",
        ],
        "Value": [
            str(BASELINE_RESULTS_DIR),
            str(INJECTED_INDEX_PATH),
            "ratio_0p00050",
            f"{inject_idx.size:,}",
            ", ".join(SCENARIOS),
            ", ".join(METHOD_FILES.keys()),
            "0.10%, 0.25%, 0.50%, 1.00%, 2.00%",
            "Exact top-k selection from raw scores",
            "Higher score indicates greater anomaly",
        ],
    }
)


# ============================================================
# 9. Save results as CSV
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\nCSV file saved to:")
print(OUTPUT_CSV)


# ============================================================
# 10. Save results as an Excel workbook
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

    # Complete row-by-row results
    results_df.to_excel(
        writer,
        sheet_name="Detailed Results",
        index=False,
    )

    # Detection-rate matrix
    detection_rate_pivot.to_excel(
        writer,
        sheet_name="Detection Rate Summary",
        index=False,
    )

    # Number of injected anomalies detected
    detected_count_pivot.to_excel(
        writer,
        sheet_name="Detected Count Summary",
        index=False,
    )

    # Average performance across scenarios
    average_detection_df.to_excel(
        writer,
        sheet_name="Average Across Scenarios",
        index=False,
    )

    # Method ranking
    ranking_df.to_excel(
        writer,
        sheet_name="Method Rankings",
        index=False,
    )

    # Raw-score statistics
    score_summary_df.to_excel(
        writer,
        sheet_name="Raw Score Summary",
        index=False,
    )

    # Evaluation settings and paths
    metadata_df.to_excel(
        writer,
        sheet_name="Experimental Settings",
        index=False,
    )

    # --------------------------------------------------------
    # Apply basic Excel formatting
    # --------------------------------------------------------

    workbook = writer.book

    for worksheet in workbook.worksheets:

        # Freeze the header row
        worksheet.freeze_panes = "A2"

        # Enable filters
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        # Bold header cells
        for cell in worksheet[1]:
            cell.font = cell.font.copy(
                bold=True
            )

        # Automatically set reasonable column widths
        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0].column_letter
            )

            max_length = 0

            for cell in column_cells:
                try:
                    cell_length = len(
                        str(cell.value)
                    )
                except Exception:
                    cell_length = 0

                max_length = max(
                    max_length,
                    cell_length,
                )

            adjusted_width = min(
                max_length + 2,
                45,
            )

            worksheet.column_dimensions[
                column_letter
            ].width = adjusted_width


print("\nExcel workbook saved to:")
print(OUTPUT_EXCEL)


# ============================================================
# 11. Display selected summaries
# ============================================================

print("\n" + "=" * 75)
print("RAW-SCORE BASELINE EVALUATION COMPLETED")
print("=" * 75)

print("\nTop-2% detection-rate results:")

top_2_percent_df = results_df.loc[
    np.isclose(
        results_df["Budget_Fraction"],
        0.02,
    ),
    [
        "Scenario",
        "Method",
        "Detected_Injected",
        "Total_Injected",
        "Detection_Rate_Percent",
    ],
].copy()

print(
    top_2_percent_df.to_string(
        index=False
    )
)

print("\nOutput files:")
print("CSV  :", OUTPUT_CSV)
print("Excel:", OUTPUT_EXCEL)